# Decentralized MNIST with dEngine

[![Runtime](https://img.shields.io/badge/Runtime-~15--30%20min-orange.svg)](#)

This notebook installs **dEngine** and runs decentralized model training benchmarks on the **MNIST** dataset.

---

### Overview
1. **Environment Setup:** Install `dEngine` and dependencies.
2. **Setup and Run:** Distribute MNIST across simulated decentralized worker nodes; Execute peer-to-peer weight exchanges and updates.
4. **Analysis & Metrics:** comparison plots and experiments dashboard.

### 1. Environment

In [ ]:
!pip install ipympl
!pip install -q git+https://github.com/DecentralizedLearning/dEngine.git@sabella-dev;
!pip install -q git+https://github.com/DecentralizedLearning/notebooks.git;

from google.colab import output
output.enable_custom_widget_manager()

### 2.1. 🧪 Experiments Setup

This notebook cell initializes and prepares four comparative distributed learning configurations using the **`dengine`** framework:
- **Centralized Baseline**
- **Decentralized Averaging (DecAvg) on Barabási–Albert (BA) Graph**
- **Decentralized Averaging (DecAvg) on Erdős–Rényi (ER) Graph**
- **Federated Learning (FedAvg)**

---

#### Imports and Dependencies
* **Standard & Utilities:** `Path` for file system handling and `tqdm.auto` for progress tracking.
* **Core Framework (`dengine`):**
  * `SimulationArguments`, `load_engine`: Simulation execution and hardware management.
  * `BUILTINS`: Standard preset configurations for scenarios, datasets, and topologies.
  * `CNNMnist`: Standard convolutional neural network used across all local clients.
  * `CentralizedScenarioEngine`, `DecAvgClient`, `FederatedClient`: Engines and client models defining each paradigm.
  * **Callbacks:** Metric-tracking hooks for logging loss curves and confusion matrices at epoch and round boundaries.

---

#### Configuration Presets

Each configuration bundles built-in YAML profiles covering dataset partition (**I.I.D. MNIST**), scenario logic, and communication topology:

| Experiment Name | Paradigm | Graph Topology | Client / Scenario Engine |
| :--- | :--- | :--- | :--- |
| `mnist,Centralized` | Centralized | Single Node / Graph | `CentralizedScenarioEngine` |
| `mnist,BA,DecAvg` | Decentralized | Barabási–Albert (`BA_SMALL`) | `DecAvgClient` |
| `mnist,ER,DecAvg` | Decentralized | Erdős–Rényi (`ER_SMALL`) | `DecAvgClient` |
| `mnist,FedAvg` | Federated | Star Topology (`STAR_SMALL`) | `FederatedClient` |

---

#### Parameter Overrides

Fine-tuned settings applied on top of the built-in defaults:

* **Centralized:**
  * Runs for **10 local epochs**.
  * Logs loss dumps and per-epoch confusion matrices.
* **Decentralized (BA & ER):**
  * Uses weighted averaging (`use_weighted_avg: True`) and shared model initialization (`common_init: True`).
  * Communication rounds limited to **2 rounds** (debug/quick test setting).
  * Logs metrics per communication round and post-aggregation.
* **Federated:**
  * Uses parameter-weighted model averaging across participating clients.
  * Set to run for **20 communication rounds**.
  * Matches decentralized callback tracking.

---

#### Execution Parameters & Output Structure

* **Directories:**
  * Datasets: `./dataset`
  * Logs & Checkpoints: `./logs`
* **Output:** A list of fully compiled configuration objects (`configurations`) ready to be passed directly to the simulation runner (`load_engine`).

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

from dengine.bin.args_parser import SimulationArguments, VerbosityLevel
from dengine.config.builtins import BUILTINS
from dengine.config.utils import convert_to_nested_dict
from dengine import load_experiment_from_yamls
from dengine.bin.simulation import load_engine
from dengine.models.classifier import CNNMnist
from dengine.scenarios.centralized import CentralizedScenarioEngine
from dengine.scenarios.decentralized import DecAvgClient
from dengine.scenarios.federated import FederatedClient
from dengine.callbacks.local import (
    LossDumpCallback,
    ConfusionMatrixDumpOnEpochEndCallback,
    ConfusionMatrixDumpOnCommRoundEndCallback,
    ConfusionMatrixDumpOnAggregationEnd
)


# ╔══════════════════════════════════════════════════════════╗
# ║  BUILT IN CONFIGS                                        ║
# ╚══════════════════════════════════════════════════════════╝
FEDERATED_CONFIGS = [
    BUILTINS.CORE.SCENARIOS.FEDERATED,
    BUILTINS.CORE.DATASETS.MNIST,
    BUILTINS.CORE.GRAPH.STAR_SMALL,
    BUILTINS.CORE.PARTITIONING.IID,
]
DECENTRALIZED_BA_CONFIGS = [
    BUILTINS.CORE.SCENARIOS.DECENTRALIZED_HOMOGENOUS,
    BUILTINS.CORE.GRAPH.BA_SMALL,
    BUILTINS.CORE.DATASETS.MNIST,
    BUILTINS.CORE.PARTITIONING.IID,
]
DECENTRALIZED_ER_CONFIGS = [
    BUILTINS.CORE.SCENARIOS.DECENTRALIZED_HOMOGENOUS,
    BUILTINS.CORE.GRAPH.ER_SMALL,
    BUILTINS.CORE.DATASETS.MNIST,
    BUILTINS.CORE.PARTITIONING.IID,
]
CENTRALIZED_CONFIGS = [
    BUILTINS.CORE.GRAPH.CENTRALIZED_GRAPH,
    BUILTINS.CORE.SCENARIOS.CENTRALIZED,
    BUILTINS.CORE.DATASETS.MNIST,
    BUILTINS.CORE.PARTITIONING.IID,
]

# ╔══════════════════════════════════════════════════════════╗
# ║  Overrides                                               ║
# ╚══════════════════════════════════════════════════════════╝
CENTRALIZED_CONFIG_OVERRIDES = {
    "scenario.target": CentralizedScenarioEngine.__name__,
    "client.local_model.target": CNNMnist.__name__,
    "client.training_engine.arguments.epochs": 10,
    # ┌──────────────────────────────────┐
    # │  Callbacks                       │
    # └──────────────────────────────────┘
    "callbacks": [
        {"target": LossDumpCallback.__name__},
        {"target": ConfusionMatrixDumpOnEpochEndCallback.__name__},
    ]
}

DECENTRALIZED_CONFIG_OVERRIDES = {
    # ┌──────────────────────────────────┐
    # │  Aggregation and Scenario        │
    # └──────────────────────────────────┘
    "client.local_model.target": CNNMnist.__name__,
    "client.target": DecAvgClient.__name__,
    "scenario.arguments.common_init": True,
    # "client.arguments.include_myself": True,
    "client.arguments.use_weighted_avg": True,
    "scenario.arguments.max_communication_rounds": 2,  # (default 200)
    # ┌──────────────────────────────────┐
    # │  Callbacks                       │
    # └──────────────────────────────────┘
    "callbacks": [
        {"target": LossDumpCallback.__name__},
        {"target": ConfusionMatrixDumpOnCommRoundEndCallback.__name__},
        {"target": ConfusionMatrixDumpOnAggregationEnd.__name__}
    ]
}

FEDERATED_CONFIGS_OVERRIDES = {
    # ┌──────────────────────────────────┐
    # │  Aggregation and Scenario        │
    # └──────────────────────────────────┘
    # Note: federated scenarios can be simulated also with a decentralized engine
    #       with a star topology graph and clients having "include_myself" set to False
    "client.local_model.target": CNNMnist.__name__,
    "client.target": FederatedClient.__name__,
    "scenario.arguments.common_init": True,
    "scenario.arguments.use_weighted_avg": True,
    "scenario.arguments.max_communication_rounds": 20,  # (default 200)
    # ┌──────────────────────────────────┐
    # │  Callbacks                       │
    # └──────────────────────────────────┘
    "callbacks": [
        {"target": LossDumpCallback.__name__},
        {"target": ConfusionMatrixDumpOnCommRoundEndCallback.__name__},
        {"target": ConfusionMatrixDumpOnAggregationEnd.__name__}
    ]
}

# ╔══════════════════════════════════════════════════════════╗
# ║  Simulation                                              ║
# ╚══════════════════════════════════════════════════════════╝
simulation_args = SimulationArguments(
    gpus=[0],
    torch_num_threads=1,
    verbosity=VerbosityLevel.silent,
    seed=123,
    resume_checkpoints=False,
    sanity_check=False,
    dump_stdout=False,
    dataset_directory=Path("./dataset"),
    output_directory=Path("./logs")
)

configurations = [
    load_experiment_from_yamls(
        files=[*CENTRALIZED_CONFIGS],
        overrides=convert_to_nested_dict({
            "name": "mnist,Centralized",
            **CENTRALIZED_CONFIG_OVERRIDES,
        }),
        experiments_directory_root=str(simulation_args.output_directory.absolute()),
        seed=simulation_args.seed,
    ),
    load_experiment_from_yamls(
        files=[*DECENTRALIZED_BA_CONFIGS],
        overrides=convert_to_nested_dict({
            "name": "mnist,BA,DecAvg",
            "graph.arguments.seed": simulation_args.seed,
            **DECENTRALIZED_CONFIG_OVERRIDES,
        }),
        experiments_directory_root=str(simulation_args.output_directory.absolute()),
        seed=simulation_args.seed,
    ),
    load_experiment_from_yamls(
        files=[*DECENTRALIZED_ER_CONFIGS],
        overrides=convert_to_nested_dict({
            "name": "mnist,ER,DecAvg",
            "graph.arguments.seed": simulation_args.seed,
            **DECENTRALIZED_CONFIG_OVERRIDES,
        }),
        experiments_directory_root=str(simulation_args.output_directory.absolute()),
        seed=simulation_args.seed,
    ),
    load_experiment_from_yamls(
        files=[*FEDERATED_CONFIGS],
        overrides=convert_to_nested_dict({
            "name": "mnist,FedAvg",
            **FEDERATED_CONFIGS_OVERRIDES,
        }),
        experiments_directory_root=str(simulation_args.output_directory.absolute()),
        seed=simulation_args.seed,
    ),
]

### 2.2. Running Experiments

In [ ]:
for cfg in tqdm(configurations, desc="Experiments"):
    loaded_engine = load_engine(simulation_args, cfg, verbose=False)
    loaded_engine.run();

<br><br>

---

<br><br>

# Comparison Plots (multiple experiments visualization)

The next cell will let you select a directory of experiments. Select the **log** directory to compare all experiments in one plot.

In [ ]:
# @title
from dnotebooks.widgets import StyledExperimentSelectionWidget

experiment_selection_widget, get_selected_experiments = StyledExperimentSelectionWidget()
display(experiment_selection_widget)

The next cell will let you select the metric to visualize. Select **confusion_matrix_test.py**.

In [ ]:
# @title
from dnotebooks.widgets import ConfusionMatrixPartitionMultiSelection

selected_experiments, line_styles = zip(*get_selected_experiments())
line_styles = {exp.name: s for exp, s in zip(selected_experiments, line_styles)}
confusion_matrix_partition_selection_wg, get_confusion_matrix = ConfusionMatrixPartitionMultiSelection(selected_experiments)
confusion_matrix_partition_selection_wg

<br>

---

<br>

In [ ]:
# @title
%matplotlib widget

import matplotlib.pyplot as plt
from dnotebooks.widgets import MultiExperimentMetricsComparisonDashboard

plt.rcParams['figure.figsize'] = [8, 5]

plt.close('all')
plots = MultiExperimentMetricsComparisonDashboard(
    {x: [v] for x, v in get_confusion_matrix().items()},
    title="Comparison Plot",
    xlabel="Communication Rounds",
    linestyles=line_styles,
    # external_legend=True
)
display(plots.render())


<br><br>
---
<br><br>


## Dashboard (single experiment visualization)

The next cell will let you select a directory of experiments. Select the **log** directory to compare all experiments in one plot. Then, select one experiment to visualize. We recomend you to choose **mnist,ER,DecAvg**.


In [3]:
# @title
from dnotebooks.widgets import MultiExperimentSelection

experiment_selection_widget, get_selected_experiments = MultiExperimentSelection(limit=1)
display(experiment_selection_widget)

The next cell will let you select the metric to visualize. Select **confusion_matrix_test.py**. Ignore the "Select Delta" dropdown.

In [9]:
# @title
from dnotebooks.widgets import ConfusionMatrixPartitionDeltaSelection

selected_experiments = get_selected_experiments()
confusion_matrix_partition_selection_wg, get_confusion_matrix = ConfusionMatrixPartitionDeltaSelection(selected_experiments)
confusion_matrix_partition_selection_wg

The next cell is required to load the experiment and its metrics.

> **⚠️ Expected Warning:**  
> Since no model dump callback was added to the configuration overrides, a checkpoint-loading warning will appear below. **This warning is expected and safe to ignore.**

In [ ]:
# @title
from pathlib import Path
from dengine.analysis import Experiment
from dnotebooks.widgets import ExperimentDashboard
from dnotebooks.plots import plot_partition_distribution, plot_scatterplot_partition_distribution

selected_experiment_path = get_selected_experiments()[0]
experiment = Experiment(
    experiment_root_path=selected_experiment_path,
    dataset_root_path=Path("./datasets/")
)

try:
    confusion_matrices = get_confusion_matrix()
except:
    confusion_matrices = None

<br>

---

<br>

> 💡 **Note:** The graph is interactive—click on nodes to inspect their details!

In [ ]:
# @title
%matplotlib widget

import matplotlib.pyplot as plt

plt.close('all')

data_distr_fig, data_distr_ax = plt.subplots(figsize=(10, 5))
plot_scatterplot_partition_distribution(experiment.partitions, data_distr_ax)
data_distr_fig.tight_layout()
plt.show()

dashboard = ExperimentDashboard(
    confusion_matrices,
    experiment.partitions,
    experiment.training_engine.graph,
)
display(dashboard.render())

<br>

---

<br>